# MACE-MDP Dipoles and Polarizability Tutorial

This notebook demonstrates dipole and polarizability predictions with the **MACE-MDP** model on the first structure in the example XYZ.


## Scientific context

- Dipole moment is the first derivative of energy with respect to electric field:
  $$\mu_i = -\frac{\partial E}{\partial F_i}$$
- Polarizability is the field derivative of dipole (equivalently second field derivative of energy):
  $$\alpha_{ij}=\frac{\partial \mu_i}{\partial F_j}=-\frac{\partial^2 E}{\partial F_i\partial F_j}$$

Connection to spectroscopy:
- IR intensity is controlled by normal-mode dipole derivatives $\partial\boldsymbol\mu/\partial Q_k$.
- Raman activity is controlled by normal-mode polarizability derivatives $\partial\boldsymbol\alpha/\partial Q_k$.

This notebook evaluates $\boldsymbol\mu$ and $\boldsymbol\alpha$ directly, then estimates simple coordinate derivatives by finite differences.

Note: Units for  Dipole ($e\ \text{Å}$) | Polarizability ($e\ \text{Å}^2/\text{V}$)


In [1]:
import numpy as np

from ase.io import read
from mace.calculators.mace import MACECalculator


/work/mace_alpha_venv/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## Paths and setup

This tutorial uses the MACE-MDP model at `../models/MACE-MDP.model`.


In [2]:
xyz_path = "../mini_database_IR-R-7193_wB97MD3.xyz"
model_path = "../../models/MACE-MDP.model"

atoms = read(xyz_path, index=0)
print(f"Using structure with {len(atoms)} atoms")
print(f"Model path: {model_path}")


Using structure with 14 atoms
Model path: ../../models/MACE-MDP.model


In [3]:
print(atoms.info)

{'name': 'HCNOFClBr/C100005', 'config_type': 'wB97M-D3(BJ)/def2-TZVPP geometries'}


## Evaluate dipole and polarizability


In [4]:
device = "cpu"  # switch to "cuda" if available

calc = MACECalculator(
    model_paths=model_path,
    model_type="DipolePolarizabilityMACE",
    default_dtype="float64",
    device=device,
)

mu = calc.get_property("dipole", atoms)
alpha = calc.get_property("polarizability", atoms)

print("Dipole vector:")
print(mu)
print("\nPolarizability tensor:")
print(alpha)


/work/mace_alpha_venv/lib/python3.11/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Dipole vector:
[ 0.23896186 -0.55568884  0.14771326]

Polarizability tensor:
[[ 1.12197371 -0.14627035  0.04957207]
 [-0.14627035  1.37639071 -0.20365442]
 [ 0.04957207 -0.20365442  0.6522868 ]]


## Interpreting the output

- `mu` is a 3-vector: molecular dipole components.
- `alpha` is a 3x3 tensor: polarizability response.